missing_link_logo_final_crop_small.png

## Pixal3D — Pixel-Aligned 3D Generation (SIGGRAPH 2026)

**Tencent's state-of-the-art** image-to-3D model. Generates high-fidelity GLB meshes with PBR textures from a single image. Built on the TRELLIS.2 backbone with pixel-aligned back-projection conditioning.

### Quick Start
1. **Connect to GPU Runtime** (L4, A100, or T4)
2. **Run All** — Cell 1 installs prebuilt CUDA wheels (~30 sec) + compiles natten (~5-10 min). flash-attn optional (skip for SDPA fallback).
3. A Gradio UI launches — click the `.gradio.live` link to open it

**Optional fast path:** MissingLink subscribers can swap Cell 1's compile section for prebuilt wheels (~2 min). Uncomment the token lines at the top of Cell 1.

### GPU Modes
| GPU | VRAM | Resolution | Mode |
|-----|------|-----------|------|
| A100 | 40 GB | 1536px | Standard |
| L4 | 24 GB | 1536px | Standard |
| T4 | 15 GB | 1024px | Low-VRAM (auto) |

### Batch Processing
Upload images to `/content/images_in/`, then use Cell 7 to batch-process without the UI.

In [ ]:
# =====================================================================
# CELL 1 — Install Dependencies (FREE — no token needed)
# =====================================================================
# Uses free public prebuilt wheels where available.
# Only natten + flash-attn compiled from source (~10-25 min total).
#
# OPTIONAL FAST PATH (MissingLink subscribers, ~2 min total):
#   Uncomment below and comment out the "BUILD FROM SOURCE" block:
#   MACHINE = 'a100'
#   TOKEN = "your_token_here"
#   !pip install --no-deps -r "https://{TOKEN}@missinglink.build/{MACHINE}.txt"
# =====================================================================
import os, sys

# Mount Google Drive
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

# Update libstdc++ for the cu130 wheels (needs GCC 11+ ABI)
!add-apt-repository ppa:ubuntu-toolchain-r/test -y && apt-get update -qq && apt-get install --only-upgrade libstdc++6 -y -qq

# --- Step 1: Python dependencies (instant, no compilation) ---
!pip install -q trimesh pygltflib plyfile moderngl huggingface_hub kornia kornia-rs \
    imageio imageio-ffmpeg tqdm easydict opencv-python-headless \
    zstandard timm diffusers accelerate gradio einops onnxruntime

# MoGe for camera estimation
!pip install -q git+https://github.com/microsoft/MoGe.git

# ====================================================================
# --- Step 2: CUDA kernels — FREE PREBUILT WHEELS ---
# ====================================================================
import torch, os, ctypes, glob
gpu_name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
arch = f"{cap[0]}{cap[1]}"
print(f"[Setup] {gpu_name} (sm_{arch}) | PyTorch {torch.__version__} | CUDA {torch.version.cuda}")

# cu130 wheels need newer libstdc++ than Colab's system one.
# PyTorch bundles GCC 11+ libs — use LD_PRELOAD to force them for all processes,
# including Gradio workers that wouldn't inherit ctypes preloads.
torch_lib = os.path.join(os.path.dirname(torch.__file__), 'lib')
torch_libstdcxx = os.path.join(torch_lib, 'libstdc++.so.6')
if os.path.exists(torch_libstdcxx):
    os.environ['LD_PRELOAD'] = torch_libstdcxx
    print(f"[Libs] LD_PRELOAD={torch_libstdcxx}")

# Use the official Pixal3D prebuilt wheels from TencentARC's storage.
WHEEL_BASE = "https://github.com/LDYang694/Storages/releases/download/rtxpro6000"
!pip install --no-deps -q \
    "{WHEEL_BASE}/cumesh-0.0.1%2Btorch2.11.0.cu130-cp312-cp312-linux_x86_64.whl" \
    "{WHEEL_BASE}/flex_gemm-1.0.0%2Btorch2.11.0.cu130-cp312-cp312-linux_x86_64.whl" \
    "{WHEEL_BASE}/o_voxel-0.0.1%2Btorch2.11.0.cu130-cp312-cp312-linux_x86_64.whl" \
    "{WHEEL_BASE}/nvdiffrast-0.4.0%2Btorch2.11.0.cu130-cp312-cp312-linux_x86_64.whl" \
    "{WHEEL_BASE}/nvdiffrec_render-0.0.0%2Btorch2.11.0.cu130-cp312-cp312-linux_x86_64.whl"
print("  cumesh, flex_gemm, o_voxel, nvdiffrast, nvdiffrec_render — done")

# --- Step 3: flash-attn (OPTIONAL — SKIPPED by default) ---
# FlashAttention speeds up inference ~10-20% but requires 15-20 min compile.
# We skip it by default — PyTorch SDPA works fine, just slightly slower.
# TO ENABLE: uncomment the 2 lines below and comment the print line.
#   !MAX_JOBS=4 pip install flash-attn --no-build-isolation -q
#   import flash_attn; print("  flash-attn — done")
print("  flash-attn — skipped (using PyTorch SDPA fallback — works fine)")

# --- Step 4: natten (OPTIONAL — may fail on newer CUDA) ---
# natten is only needed for NAF upsampling (shape/texture stages).
# If the install below fails, Cell 3 will auto-disable NAF and use
# DINO features directly — quality is slightly lower but it works.
natten_ok = False
try:
    print(f"[Build] Compiling natten for sm_{arch} (~5-10 min)…")
    !NATTEN_CUDA_ARCH="{arch}" NATTEN_N_WORKERS=4 pip install natten==0.21.0 --no-build-isolation -q 2>/dev/null
    import natten
    _has_libnatten = getattr(natten, 'HAS_LIBNATTEN', False)
    natten_ok = _has_libnatten
    print(f"  natten — done (libnatten={'OK' if _has_libnatten else 'NONE'})")
except Exception as e:
    stderr = str(e).splitlines()[0][:200]
    print(f"  natten — failed ({stderr})")
    print("  NAF upsampling disabled — DINO features used directly")

# --- Step 5: utils3d (pure Python, instant) ---
!pip install -q --force-reinstall --no-deps \
    https://github.com/LDYang694/Storages/releases/download/20260430/utils3d-0.0.2-py3-none-any.whl

# --- CUDA aliases (handle suffixed package names) ---
import sys, importlib
for base, names in [('cumesh', ('cumesh_vb','cumesh')),
                    ('flex_gemm', ('flex_gemm_ap','flex_gemm')),
                    ('o_voxel', ('o_voxel_vb_ap','o_voxel'))]:
    for name in names:
        try:
            m = importlib.import_module(name)
            sys.modules.setdefault(base, m)
            break
        except ImportError: pass

os.makedirs("/content/images_in", exist_ok=True)

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"\n[DONE] {gpu_name} ({vram_gb:.1f} GB) — all dependencies installed")

### Cell 2 — Clone Repo & Cache Models (~3-5 min)

Clones Pixal3D and pre-downloads model weights from HuggingFace. Models download only once — subsequent runs skip this step.

In [ ]:
# =====================================================================
# CELL 2 — Clone Repo & Pre-cache Models
# =====================================================================
import os, torch, subprocess

os.chdir("/content")

# Clone Pixal3D (main branch = TRELLIS.2 backbone)
if not os.path.exists("/content/Pixal3D/.git"):
    if os.path.exists("/content/Pixal3D"):
        import shutil
        shutil.rmtree("/content/Pixal3D")
    subprocess.run(["git", "clone", "https://github.com/TencentARC/Pixal3D.git", "/content/Pixal3D"], check=True)
else:
    print("[Clone] Pixal3D already cloned")

os.chdir("/content/Pixal3D")
subprocess.run(["git", "lfs", "pull"], capture_output=True)

from huggingface_hub import snapshot_download

# Optional: set HF token for faster downloads
# from huggingface_hub import login; login(token="hf_xxx")

print("[Cache] DinoV3 feature extractor...")
snapshot_download("camenduru/dinov3-vitl16-pretrain-lvd1689m",
    allow_patterns=["*.bin", "*.json", "*.txt", "*.model", "*.safetensors"],
    max_workers=4)

print("[Cache] BiRefNet (background removal)...")
snapshot_download("ZhengPeng7/BiRefNet",
    allow_patterns=["*.pth", "*.json", "*.txt", "*.bin", "*.model"],
    max_workers=2)

# Verify HDRI assets exist
hdri_dir = "/content/Pixal3D/assets/hdri"
if os.path.isdir(hdri_dir):
    exr_files = [f for f in os.listdir(hdri_dir) if f.endswith('.exr')]
    if exr_files:
        print(f"[Assets] {len(exr_files)} HDRI maps found")
    else:
        print("[Warning] No .exr files — preview renders will use fallback")

print("\n[DONE] All models cached. Pixal3D weights auto-download on first inference.")

os.makedirs("/content/Pixal3D/output", exist_ok=True)

### Cell 3 — Environment Setup & Core Functions

Sets environment variables, imports, and defines the model initialization and inference functions. Run this before either the Gradio UI or batch processing.

In [ ]:
# =====================================================================
# CELL 3 — Environment & Core Functions
# =====================================================================
import os, sys, math, time, json, glob, gc
import torch, numpy as np
from PIL import Image
from datetime import datetime

# --- Environment ---
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Fix: cu130 wheels need newer libstdc++ than Colab's system version.
# PyTorch bundles GCC 11+ compatible libs — preload them with ctypes
# BEFORE importing cv2 (which would lock the old libstdc++).
import ctypes
torch_lib = os.path.join(os.path.dirname(torch.__file__), 'lib')
os.environ['LD_LIBRARY_PATH'] = torch_lib + ':' + os.environ.get('LD_LIBRARY_PATH', '/usr/lib/x86_64-linux-gnu')
for libname in ['libstdc++.so.6', 'libgcc_s.so.1', 'libc10_cuda.so', 'libtorch_cuda.so']:
    path = os.path.join(torch_lib, libname)
    if os.path.exists(path):
        ctypes.CDLL(path, mode=ctypes.RTLD_GLOBAL)
        print(f"[Libs] Preloaded: {libname}")

import cv2  # Must go AFTER libstdc++ preload

# Auto-detect best attention backend: FlashAttention 3 -> FA2 -> sdpa
attn_backend = 'sdpa'
try:
    import flash_attn_interface
    attn_backend = 'flash_attn_3'
    print("[Attn] Using FlashAttention 3 (flash_attn_interface)")
except ImportError:
    try:
        import flash_attn
        attn_backend = 'flash_attn'
        print("[Attn] Using FlashAttention 2 (flash_attn)")
    except ImportError:
        print("[Attn] Falling back to PyTorch SDPA (no FlashAttention)")
os.environ["ATTN_BACKEND"] = attn_backend
os.environ["SPARSE_ATTN_BACKEND"] = attn_backend  # Also set for sparse modules
os.environ["FLEX_GEMM_AUTOTUNE_CACHE_PATH"] = "/content/autotune_cache.json"
os.environ["FLEX_GEMM_AUTOTUNER_VERBOSE"] = '1'

os.chdir("/content/Pixal3D")
if "/content/Pixal3D" not in sys.path:
    sys.path.insert(0, "/content/Pixal3D")

# --- Constants ---
MODEL_PATH = "TencentARC/Pixal3D"
MOGE_MODEL_NAME = "Ruicheng/moge-2-vitl"
WILD_MESH_SCALE = 1.0
WILD_EXTEND_PIXEL = 0
WILD_IMAGE_RESOLUTION = 512

IMAGE_COND_CONFIGS = {
    "ss": {"model_name": "camenduru/dinov3-vitl16-pretrain-lvd1689m", "image_size": 512,  "grid_resolution": 16},
    "shape_512":  {"model_name": "camenduru/dinov3-vitl16-pretrain-lvd1689m", "image_size": 512,  "grid_resolution": 32,  "use_naf_upsample": True, "naf_target_size": 512},
    "shape_1024": {"model_name": "camenduru/dinov3-vitl16-pretrain-lvd1689m", "image_size": 1024, "grid_resolution": 64,  "use_naf_upsample": True, "naf_target_size": 512},
    "tex_1024":   {"model_name": "camenduru/dinov3-vitl16-pretrain-lvd1689m", "image_size": 1024, "grid_resolution": 64,  "use_naf_upsample": True, "naf_target_size": 1024},
}

# If natten failed to install in Cell 1, disable NAF upsampling
if not globals().get('natten_ok', False):
    print("[NAF] NATTEN unavailable — disabling NAF upsampling (DINO features used directly)")
    for k in ('shape_512', 'shape_1024', 'tex_1024'):
        IMAGE_COND_CONFIGS[k] = {**IMAGE_COND_CONFIGS[k], 'use_naf_upsample': False}

# --- Globals ---
pipeline = None
moge_model = None
envmap = None
_low_vram_active = None

# =========================================================================
# Model Helpers
# =========================================================================

def build_image_cond_model(config):
    from pixal3d.trainers.flow_matching.mixins.image_conditioned_proj import DinoV3ProjFeatureExtractor
    m = DinoV3ProjFeatureExtractor(**config)
    m.eval()
    return m

def load_moge_model(device="cuda"):
    from moge.model.v2 import MoGeModel
    m = MoGeModel.from_pretrained(MOGE_MODEL_NAME).to(device)
    m.eval()
    return m

def cleanup_memory():
    """Aggressive CUDA memory cleanup — call between batch items or after OOM."""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
# =========================================================================
# Init All Models
# =========================================================================

def init_models(low_vram=False):
    global pipeline, moge_model, envmap, _low_vram_active
    if pipeline is not None and _low_vram_active == low_vram:
        return
    _low_vram_active = low_vram

    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    gpu_name = torch.cuda.get_device_name(0)
    print(f"{'='*60}")
    print(f"[Init] {gpu_name} ({vram:.1f} GB) | PyTorch {torch.__version__} | Low-VRAM: {low_vram}")
    print(f"{'='*60}")

    from pixal3d.pipelines import Pixal3DImageTo3DPipeline

    print(f"[Pipeline] Loading {MODEL_PATH}...")
    pipeline = Pixal3DImageTo3DPipeline.from_pretrained(MODEL_PATH)

    print("[ImageCond] Building DinoV3 models...")
    pipeline.image_cond_model_ss = build_image_cond_model(IMAGE_COND_CONFIGS["ss"])
    pipeline.image_cond_model_shape_512 = build_image_cond_model(IMAGE_COND_CONFIGS["shape_512"])
    pipeline.image_cond_model_shape_1024 = build_image_cond_model(IMAGE_COND_CONFIGS["shape_1024"])
    pipeline.image_cond_model_tex_1024 = build_image_cond_model(IMAGE_COND_CONFIGS["tex_1024"])

    if low_vram:
        print("[NAF] Pre-downloading weights (CPU only)...")
        for a in ['image_cond_model_ss','image_cond_model_shape_512','image_cond_model_shape_1024','image_cond_model_tex_1024']:
            m = getattr(pipeline, a, None)
            if m is not None and getattr(m, 'use_naf_upsample', False):
                m._load_naf()
        pipeline._device = torch.device("cuda")
        pipeline.low_vram = True
    else:
        pipeline.low_vram = False
        pipeline.cuda()
        pipeline.image_cond_model_ss.cuda()
        pipeline.image_cond_model_shape_512.cuda()
        pipeline.image_cond_model_shape_1024.cuda()
        pipeline.image_cond_model_tex_1024.cuda()
        print("[NAF] Pre-loading upsampler...")
        for a in ['image_cond_model_ss','image_cond_model_shape_512','image_cond_model_shape_1024','image_cond_model_tex_1024']:
            m = getattr(pipeline, a, None)
            if m is not None and getattr(m, 'use_naf_upsample', False):
                m._load_naf()

    print("[MoGe-2] Loading camera model...")
    moge_model = load_moge_model(device="cpu" if low_vram else "cuda")

    print("[EnvMap] Loading HDRI...")
    from pixal3d.renderers import EnvMap
    dev = 'cpu' if low_vram else 'cuda'
    envmap = {}
    _base = "/content/Pixal3D/assets/hdri"
    for name in ['forest','sunset','courtyard']:
        p = os.path.join(_base, f'{name}.exr')
        if os.path.exists(p):
            envmap[name] = EnvMap(torch.tensor(
                cv2.cvtColor(cv2.imread(p, cv2.IMREAD_UNCHANGED), cv2.COLOR_BGR2RGB),
                dtype=torch.float32, device=dev))
    print(f"[Init] {len(envmap)} HDRI maps loaded. Ready!")

# =========================================================================
# Camera Helpers
# =========================================================================

def compute_f_pixels(camera_angle_x, resolution):
    fl = 16.0 / torch.tan(torch.tensor(camera_angle_x / 2.0))
    return float((fl * resolution / 32.0).item())

def distance_from_fov(camera_angle_x, grid_point, target_point, mesh_scale, image_resolution):
    R = torch.tensor([[1.,0.,0.],[0.,0.,-1.],[0.,1.,0.]])
    gp = grid_point.float() @ R.T / mesh_scale / 2
    xt, yt = float(target_point[0].item()), float(target_point[1].item())
    fp = compute_f_pixels(camera_angle_x, image_resolution)
    return {"distance_from_x": float(fp * gp[0].item() / (xt - image_resolution/2) - gp[1].item()), "f_pixels": fp}

def get_camera_params(image_path, moge, device="cuda", mesh_scale=1.0, extend_pixel=0, image_resolution=512):
    global _low_vram_active
    if _low_vram_active:
        moge.to(device)
    img = Image.open(image_path).convert("RGB")
    w, h = img.size
    t = torch.from_numpy(np.array(img).astype(np.float32)/255.).permute(2,0,1).to(device)
    with torch.no_grad():
        out = moge.infer(t)
    if _low_vram_active:
        moge.cpu(); cleanup_memory()
    fx_norm = out["intrinsics"].squeeze().cpu().numpy()[0,0]
    camera_angle_x = 2 * math.atan(w / (2 * fx_norm * w))
    d = distance_from_fov(camera_angle_x, torch.tensor([-1.,0.,0.]),
        torch.tensor([0-extend_pixel, image_resolution-1+extend_pixel]), mesh_scale, image_resolution)
    return {'camera_angle_x': camera_angle_x, 'distance': d['distance_from_x'], 'mesh_scale': mesh_scale}

# =========================================================================
# Core Inference (no Gradio dependency — usable from batch/gradio/cli)
# =========================================================================

def run_inference(image_path, seed=42, resolution=1536, low_vram=False,
                  ss_steps=12, ss_guidance=7.5, ss_rescale=0.7, ss_rescale_t=5.0,
                  shape_steps=12, shape_guidance=7.5, shape_rescale=0.5, shape_rescale_t=3.0,
                  tex_steps=12, tex_guidance=1.0, tex_rescale=0.0, tex_rescale_t=3.0,
                  output_path=None, render_preview=False, verbose=True):
    """
    Core inference — callable from Gradio UI or batch loop.
    Returns: (glb_path, preview_png_path_or_None)
    """
    global pipeline, moge_model, envmap

    t0 = time.time()

    if pipeline is None:
        init_models(low_vram=low_vram)

    if verbose: print(f"[Inference] {os.path.basename(image_path)}")

    # Preprocess (background removal + crop)
    img = Image.open(image_path).convert("RGBA")
    image_preprocessed = pipeline.preprocess_image(img)

    # Camera estimation
    tmp = "/content/_tmp_cam.png"
    image_preprocessed.save(tmp)
    cam = get_camera_params(tmp, moge_model, mesh_scale=1.0, extend_pixel=0, image_resolution=512)
    os.remove(tmp)
    if verbose: print(f"  Camera: fov={math.degrees(cam['camera_angle_x']):.1f}°, dist={cam['distance']:.3f}")

    # Run pipeline
    torch.manual_seed(int(seed))
    ss = {"steps": int(ss_steps), "guidance_strength": float(ss_guidance),
          "guidance_rescale": float(ss_rescale), "rescale_t": float(ss_rescale_t)}
    sh = {"steps": int(shape_steps), "guidance_strength": float(shape_guidance),
          "guidance_rescale": float(shape_rescale), "rescale_t": float(shape_rescale_t)}
    tx = {"steps": int(tex_steps), "guidance_strength": float(tex_guidance),
          "guidance_rescale": float(tex_rescale), "rescale_t": float(tex_rescale_t)}

    pipe_type = f"{int(resolution)}_cascade"
    if verbose: print(f"  Pipeline: {pipe_type}")

    mesh_list, (shape_slat, tex_slat, res) = pipeline.run(
        image_preprocessed, camera_params=cam, seed=int(seed),
        sparse_structure_sampler_params=ss,
        shape_slat_sampler_params=sh,
        tex_slat_sampler_params=tx,
        preprocess_image=False, return_latent=True,
        pipeline_type=pipe_type, max_num_tokens=49152)

    # Extract GLB
    import o_voxel
    mesh = mesh_list[0]
    glb = o_voxel.postprocess.to_glb(
        vertices=mesh.vertices, faces=mesh.faces, attr_volume=mesh.attrs,
        coords=mesh.coords, attr_layout=pipeline.pbr_attr_layout,
        grid_size=res, aabb=[[-0.5,-0.5,-0.5],[0.5,0.5,0.5]],
        decimation_target=1000000, texture_size=4096,
        remesh=True, remesh_band=1, remesh_project=0, use_tqdm=True)

    rot = np.array([[-1,0,0,0],[0,0,-1,0],[0,-1,0,0],[0,0,0,1]], dtype=np.float64)
    glb.apply_transform(rot)

    if output_path is None:
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        name = os.path.splitext(os.path.basename(image_path))[0]
        output_path = f"/content/Pixal3D/output/{name}_{ts}.glb"
    os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
    glb.export(output_path, extension_webp=False)  # False = wider 3D viewer compat

    # Optional preview render
    preview_path = None
    if render_preview:
        try:
            from pixal3d.utils import render_utils
            from pixal3d.renderers import EnvMap
            cd = cam['distance']
            has_env = bool(envmap)
            if has_env:
                if low_vram:
                    for v in envmap.values():
                        v.image = v.image.cuda()
                        if hasattr(v, '_nvdiffrec_envlight'): del v._nvdiffrec_envlight
                mesh.simplify(16777216)
                renders = render_utils.render_proj_aligned_video(
                    mesh, camera_angle_x=cam['camera_angle_x'], distance=cd,
                    resolution=512, num_frames=8, envmap=envmap,
                    near=max(0.01, cd-2.0), far=cd+10.0)
                if low_vram:
                    for v in envmap.values():
                        if hasattr(v, '_nvdiffrec_envlight'): del v._nvdiffrec_envlight
                        v.image = v.image.cpu()
                    torch.cuda.empty_cache()
                k = 'shaded_forest' if 'shaded_forest' in renders else list(renders.keys())[0]
                frames = renders[k]
                preview_path = output_path.replace('.glb','_preview.png')
                Image.fromarray(frames[len(frames)//2]).save(preview_path)
        except Exception as e:
            if verbose: print(f"  [Preview skip] {e}")

    elapsed = time.time() - t0
    if verbose: print(f"  [Done] {output_path} ({elapsed:.0f}s)")

    # Free memory between batch items
    if low_vram:
        cleanup_memory()

    return output_path, preview_path

print("[Setup] Core functions ready. Run Cell 4 for Gradio UI or Cell 7 for batch processing.")

### Cell 4 — Launch Gradio UI

Interactive web interface for single-image 3D generation. Click the `.gradio.live` link to open in a separate tab.

In [ ]:
# =====================================================================
# CELL 4 — Gradio UI
# =====================================================================
import sys, os
if "/content/Pixal3D" not in sys.path:
    sys.path.insert(0, "/content/Pixal3D")
print(f"[Gradio] sys.path includes Pixal3D: {'/content/Pixal3D' in sys.path}")

import gradio as gr

vram_total = torch.cuda.get_device_properties(0).total_memory / 1024**3
gpu_name = torch.cuda.get_device_name(0)
default_lv = vram_total < 20
default_res = 1024 if default_lv else 1536

def generate_wrapper(image_input, seed, resolution, low_vram,
                     ss_steps, ss_guidance, ss_rescale, ss_rescale_t,
                     shape_steps, shape_guidance, shape_rescale, shape_rescale_t,
                     tex_steps, tex_guidance, tex_rescale, tex_rescale_t,
                     progress=gr.Progress()):
    import sys, os, ctypes, torch
    if "/content/Pixal3D" not in sys.path:
        sys.path.insert(0, "/content/Pixal3D")
    # Gradio workers don't inherit library preloads — redo them here.
    torch_lib = os.path.join(os.path.dirname(torch.__file__), 'lib')
    for libname in ['libstdc++.so.6', 'libgcc_s.so.1', 'libc10_cuda.so', 'libtorch_cuda.so']:
        path = os.path.join(torch_lib, libname)
        if os.path.exists(path):
            ctypes.CDLL(path, mode=ctypes.RTLD_GLOBAL)
    global pipeline
    progress(0, desc="Initializing...")
    if pipeline is None:
        init_models(low_vram=low_vram)
    progress(0.3, desc="Generating 3D...")
    glb_path, prev_path = run_inference(
        image_input, seed=seed, resolution=resolution, low_vram=low_vram,
        ss_steps=ss_steps, ss_guidance=ss_guidance, ss_rescale=ss_rescale, ss_rescale_t=ss_rescale_t,
        shape_steps=shape_steps, shape_guidance=shape_guidance, shape_rescale=shape_rescale, shape_rescale_t=shape_rescale_t,
        tex_steps=tex_steps, tex_guidance=tex_guidance, tex_rescale=tex_rescale, tex_rescale_t=tex_rescale_t,
        output_path=None, render_preview=True, verbose=True)
    progress(0.9, desc="Done!")
    return glb_path, prev_path

with gr.Blocks(title="Pixal3D — MissingLink", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Pixal3D — Pixel-Aligned 3D Generation\nTencent SIGGRAPH 2026. Image-to-3D with PBR textures.")
    with gr.Row():
        with gr.Column(scale=1):
            img_in = gr.Image(type="filepath", label="Input Image", height=280)
            with gr.Accordion("Settings", open=True):
                seed = gr.Number(label="Seed", value=42, precision=0)
                resolution = gr.Radio(["1024","1536"], label="Resolution", value=str(default_res))
                low_vram = gr.Checkbox(label="Low VRAM Mode", value=default_lv)
            with gr.Accordion("Advanced Sampling", open=False):
                gr.Markdown("**Sparse Structure**")
                ss_steps = gr.Slider(4, 20, value=12, step=1, label="Steps"); ss_cfg = gr.Slider(1.0, 15.0, value=7.5, step=0.5, label="CFG")
                ss_resc = gr.Slider(0.0, 1.0, value=0.7, step=0.1, label="Rescale"); ss_rt = gr.Slider(1.0, 10.0, value=5.0, step=0.5, label="Rescale T")
                gr.Markdown("**Shape**")
                sh_steps = gr.Slider(4, 20, value=8 if default_lv else 12, step=1, label="Steps"); sh_cfg = gr.Slider(1.0, 15.0, value=7.5, step=0.5, label="CFG")
                sh_resc = gr.Slider(0.0, 1.0, value=0.5, step=0.1, label="Rescale"); sh_rt = gr.Slider(1.0, 10.0, value=3.0, step=0.5, label="Rescale T")
                gr.Markdown("**Texture**")
                tx_steps = gr.Slider(4, 20, value=8 if default_lv else 12, step=1, label="Steps"); tx_cfg = gr.Slider(1.0, 15.0, value=1.0, step=0.5, label="CFG")
                tx_resc = gr.Slider(0.0, 1.0, value=0.0, step=0.1, label="Rescale"); tx_rt = gr.Slider(1.0, 10.0, value=3.0, step=0.5, label="Rescale T")
            btn = gr.Button("Generate 3D Mesh", variant="primary", size="lg")
        with gr.Column(scale=1):
            preview = gr.Image(label="Turntable Preview", height=280)
            out_file = gr.File(label="Download GLB", file_types=[".glb"])
            gr.Markdown(f"**GPU:** {gpu_name} ({vram_total:.0f} GB) | **Mode:** {'Low-VRAM' if default_lv else 'Standard'} | Outputs: `Pixal3D/output/`")
    btn.click(fn=generate_wrapper,
        inputs=[img_in,seed,resolution,low_vram,ss_steps,ss_cfg,ss_resc,ss_rt,sh_steps,sh_cfg,sh_resc,sh_rt,tx_steps,tx_cfg,tx_resc,tx_rt],
        outputs=[out_file,preview])
    gr.Markdown("**Tips:** Clean-background images work best. Lower steps = faster. GLBs include PBR textures.")

demo.queue(max_size=3).launch(share=True, debug=False, show_error=True)

### Cell 5 — Keep Alive (Optional)
Prevents Colab from disconnecting during long inference runs.

In [ ]:
import threading, time
def keep_alive_fn():
    while True:
        time.sleep(300)  # Ping every 5 minutes
        print("\u200b", end="", flush=True)  # Zero-width space — keeps session alive without cluttering output
try:
    t = threading.Thread(target=keep_alive_fn, daemon=True); t.start()
    print("[KeepAlive] Active (invisible pings every 5 min).")
except Exception as e:
    print(f"[KeepAlive] Error: {e}")

### Cell 6 — Single Image Inference (CLI-style)

Generate one GLB from a single image without using the Gradio UI. Good for testing.

In [ ]:
# =====================================================================
# CELL 6 — Single Image Inference (no UI)
# =====================================================================
# Replace with your image path:
IMAGE_PATH = "/content/Pixal3D/assets/images/1_img.png"

if os.path.exists(IMAGE_PATH):
    glb_path, preview_path = run_inference(
        IMAGE_PATH, seed=42, resolution=1536, low_vram=default_lv,
        ss_steps=12, ss_guidance=7.5, ss_rescale=0.7, ss_rescale_t=5.0,
        shape_steps=8 if default_lv else 12, shape_guidance=7.5, shape_rescale=0.5, shape_rescale_t=3.0,
        tex_steps=8 if default_lv else 12, tex_guidance=1.0, tex_rescale=0.0, tex_rescale_t=3.0,
        render_preview=True, verbose=True)
    if preview_path:
        from IPython.display import display, Image as IPImage
        display(IPImage(filename=preview_path))
else:
    print(f"Image not found: {IMAGE_PATH}")
    print("Upload images to /content/images_in/ or use a built-in test image.")

### Cell 7 — Batch Process Images

Upload images to `/content/images_in/` (via Colab file browser), then run this cell. Outputs go to Google Drive.

Each GLB is named `{image_name}_{timestamp}.glb` with a preview render.

In [ ]:
# =====================================================================
# CELL 7 — Batch Process All Images in /content/images_in/
# =====================================================================
import shutil
from datetime import datetime

INPUT_DIR = "/content/images_in"
OUTPUT_DIR = "/content/drive/MyDrive/pixal3d_batch_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Collect images
images = []
for ext in ['*.png', '*.jpg', '*.jpeg', '*.webp', '*.PNG', '*.JPG', '*.JPEG']:
    images.extend(glob.glob(os.path.join(INPUT_DIR, ext)))
images = sorted(set(images))

if not images:
    print(f"[Batch] No images found in {INPUT_DIR}/")
    print("  Upload images using the Colab file browser (left sidebar) -> /content/images_in/")
else:
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    lv = vram < 20
    res = 1024 if lv else 1536
    print(f"[Batch] {len(images)} images | GPU: {torch.cuda.get_device_name(0)} | Resolution: {res}px | Low-VRAM: {lv}")
    print(f"[Batch] Output: {OUTPUT_DIR}/\n")

    init_models(low_vram=lv)  # Load once for full batch

    results = []
    for i, img_path in enumerate(images):
        name = os.path.splitext(os.path.basename(img_path))[0]
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        out_path = os.path.join(OUTPUT_DIR, f"{name}_{ts}.glb")

        # Skip if already processed (same basename, any timestamp)
        existing = glob.glob(os.path.join(OUTPUT_DIR, f"{name}_*.glb"))
        if existing:
            print(f"  [{i+1}/{len(images)}] {name} — SKIP (already exists)")
            results.append((name, "skipped", existing[0]))
            continue

        print(f"  [{i+1}/{len(images)}] {name} — generating...")
        try:
            glb_path, prev_path = run_inference(
                img_path, seed=42 + i, resolution=res, low_vram=lv,
                ss_steps=12, ss_guidance=7.5, ss_rescale=0.7, ss_rescale_t=5.0,
                shape_steps=8 if lv else 12, shape_guidance=7.5, shape_rescale=0.5, shape_rescale_t=3.0,
                tex_steps=8 if lv else 12, tex_guidance=1.0, tex_rescale=0.0, tex_rescale_t=3.0,
                render_preview=True, verbose=False)
            # Move to Drive
            shutil.move(glb_path, out_path)
            if prev_path:
                prev_out = out_path.replace('.glb','_preview.png')
                shutil.move(prev_path, prev_out)
            results.append((name, "ok", out_path))
        except Exception as e:
            print(f"    [ERROR] {e}")
            results.append((name, "error", str(e)))
            cleanup_memory()

    # Summary
    ok = sum(1 for r in results if r[1] == 'ok')
    sk = sum(1 for r in results if r[1] == 'skipped')
    er = sum(1 for r in results if r[1] == 'error')
    print(f"\n[Batch] Done! {ok} generated, {sk} skipped, {er} errors")
    print(f"[Batch] Output directory: {OUTPUT_DIR}/")